In [89]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [90]:
tf.random.set_seed(42)
np.random.seed(42)
 

In [91]:
data = {
    "text": [
        "I absolutely loved this movie, it was fantastic",
        "What a wonderful and heartwarming story",
        "Best film I have seen in years, truly amazing",
        "The acting was superb and the plot was gripping",
        "I enjoyed every single minute of it",
        "This movie was terrible and a complete waste of time",
        "I hated the plot, it made no sense at all",
        "Worst film ever, painfully boring throughout",
        "The acting was awful and the story was dull",
        "I would never recommend this movie to anyone",
        "This was a great and enjoyable movie",
        "This was a boring and terrible film"
    ],
    "label": [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 0]
}
 
df = pd.DataFrame(data)
print("Dummy Dataset:")
print(df)
print("\n" + "=" * 60 + "\n")
 

Dummy Dataset:
                                                 text  label
0     I absolutely loved this movie, it was fantastic      1
1             What a wonderful and heartwarming story      1
2       Best film I have seen in years, truly amazing      1
3     The acting was superb and the plot was gripping      1
4                 I enjoyed every single minute of it      1
5   This movie was terrible and a complete waste o...      0
6           I hated the plot, it made no sense at all      0
7        Worst film ever, painfully boring throughout      0
8         The acting was awful and the story was dull      0
9        I would never recommend this movie to anyone      0
10               This was a great and enjoyable movie      1
11                This was a boring and terrible film      0




In [92]:
VOCAB_SIZE = 1000  # max number of unique words the tokenizer will track
OOV_TOKEN = "<OOV>"  # token used for words not seen during training ("Out Of Vocabulary")
 
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
 

tokenizer.fit_on_texts(df["text"])
 

sequences = tokenizer.texts_to_sequences(df["text"])
 
print("Word Index (word -> integer id):")
print(tokenizer.word_index)
print("\nExample sentence converted to a sequence of integers:")
print(f"'{df['text'][0]}' -> {sequences[0]}")
print("\n" + "=" * 60 + "\n")

Word Index (word -> integer id):
{'<OOV>': 1, 'was': 2, 'and': 3, 'i': 4, 'this': 5, 'the': 6, 'movie': 7, 'a': 8, 'it': 9, 'film': 10, 'story': 11, 'acting': 12, 'plot': 13, 'of': 14, 'terrible': 15, 'boring': 16, 'absolutely': 17, 'loved': 18, 'fantastic': 19, 'what': 20, 'wonderful': 21, 'heartwarming': 22, 'best': 23, 'have': 24, 'seen': 25, 'in': 26, 'years': 27, 'truly': 28, 'amazing': 29, 'superb': 30, 'gripping': 31, 'enjoyed': 32, 'every': 33, 'single': 34, 'minute': 35, 'complete': 36, 'waste': 37, 'time': 38, 'hated': 39, 'made': 40, 'no': 41, 'sense': 42, 'at': 43, 'all': 44, 'worst': 45, 'ever': 46, 'painfully': 47, 'throughout': 48, 'awful': 49, 'dull': 50, 'would': 51, 'never': 52, 'recommend': 53, 'to': 54, 'anyone': 55, 'great': 56, 'enjoyable': 57}

Example sentence converted to a sequence of integers:
'I absolutely loved this movie, it was fantastic' -> [4, 17, 18, 5, 7, 9, 2, 19]




In [93]:
MAX_LEN = max(len(seq) for seq in sequences)  # length of the longest sentence
 
# padding="post" -> add zeros AFTER the sentence (e.g. [3,5,2,0,0])
# truncating="post" -> if a sentence is too long, cut it off at the end
padded_sequences = pad_sequences(
    sequences, maxlen=MAX_LEN, padding="post", truncating="post"
)
 
print(f"Max sequence length used for padding: {MAX_LEN}")
print("Padded sequences (each row = one sentence, same length now):")
print(padded_sequences)
print("\n" + "=" * 60 + "\n")
 
# Our final inputs (X) and targets (y) for the model
X = padded_sequences
y = df["label"].values
 

Max sequence length used for padding: 10
Padded sequences (each row = one sentence, same length now):
[[ 4 17 18  5  7  9  2 19  0  0]
 [20  8 21  3 22 11  0  0  0  0]
 [23 10  4 24 25 26 27 28 29  0]
 [ 6 12  2 30  3  6 13  2 31  0]
 [ 4 32 33 34 35 14  9  0  0  0]
 [ 5  7  2 15  3  8 36 37 14 38]
 [ 4 39  6 13  9 40 41 42 43 44]
 [45 10 46 47 16 48  0  0  0  0]
 [ 6 12  2 49  3  6 11  2 50  0]
 [ 4 51 52 53  5  7 54 55  0  0]
 [ 5  2  8 56  3 57  7  0  0  0]
 [ 5  2  8 16  3 15 10  0  0  0]]




In [94]:
EMBEDDING_DIM = 16   # size of each word's dense vector representation
RNN_UNITS = 32       # number of neurons in the RNN's hidden state
 
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    SimpleRNN(RNN_UNITS, activation="tanh"),   # return_sequences=False by default => Many-to-One
    Dense(1, activation="sigmoid")             # single output neuron: probability of "Positive"
])
 
# Build the model so summary() shows shapes correctly
model.build(input_shape=(None, MAX_LEN))
model.summary()
print("\n" + "=" * 60 + "\n")
 

c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ (None, 10, 16)         │        16,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_9 (SimpleRNN)        │ (None, 32)             │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,601 (68.75 KB)

 Trainable params: 17,601 (68.75 KB)

 Non-trainable params: 0 (0.00 B)

In [95]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)
 

In [96]:
EPOCHS = 30
BATCH_SIZE = 2
 
history = model.fit(
    X, y,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)
 

Epoch 1/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4167 - loss: 0.6946  
Epoch 2/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8333 - loss: 0.6569 
Epoch 3/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9167 - loss: 0.6224 
Epoch 4/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9167 - loss: 0.5798 
Epoch 5/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9167 - loss: 0.5223 
Epoch 6/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.4443 
Epoch 7/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.3462 
Epoch 8/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 1.0000 - loss: 0.2408 
Epoch 9/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.1497 
Epoch 10/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0877 
Epoch 11/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0522 
Epoch 12/30
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 1.0000 - loss: 0.0331 


In [97]:
loss, accuracy = model.evaluate(X, y, verbose=0)
print("\n" + "=" * 60)
print(f"Final Training Loss:     {loss:.4f}")
print(f"Final Training Accuracy: {accuracy:.4f}  ({accuracy * 100:.2f}%)")
print("=" * 60)
 
 


Final Training Loss:     0.0041
Final Training Accuracy: 1.0000  (100.00%)


In [98]:
def predict_sentiment(sentence: str) -> None:
    """Preprocess a new sentence the same way as training data, then predict."""
    seq = tokenizer.texts_to_sequences([sentence])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    prob = model.predict(padded, verbose=0)[0][0]
    label = "Positive" if prob >= 0.5 else "Negative"
    print(f"'{sentence}' -> {label} (confidence: {prob:.2f})")
 
 
print("\nTrying the model on new, unseen sentences:")
predict_sentiment("This was a great and enjoyable movie")
predict_sentiment("This was a boring and terrible film")
 


Trying the model on new, unseen sentences:
'This was a great and enjoyable movie' -> Positive (confidence: 0.99)
'This was a boring and terrible film' -> Negative (confidence: 0.01)
